# Relatório da matéria de "Bioinformática Aplicada à Genômica Bacteriana"
### Aluno: David Tavares Martins
### Nº de matrícula: 2024746980

Para as atividades do relatório foram escolhidos dados de Mycobacterium tuberculosis advindos de sequenciamento de nova geração
Plataforma llumina NovaSeq 6000
Serão realizadas etapas de análises genômicas com esses dados incluindo:
    Controle de Qualidade
    Montagem
    Anotação
    Filogenia
    Pangenomica
    Predição de ilhas genômicas com Gipsy

## 1ª Etapa: Baixar os dados brutos

In [ ]:
!wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/067/SRR25113367/SRR25113367_1.fastq.gz -O data/mycotuberculosis_1.fastq.gz
!wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR251/067/SRR25113367/SRR25113367_2.fastq.gz -O data/mycotuberculosis_2.fastq.gz

## 2ª Etapa: Avaliando a qualidade das leituras com fastqc

O fastqc gera gráficos mostrando como estão se comportando os dados de Sequênciamento quanto a:
    Qualidade das leituras
    Presença de adaptadores
    Presença de Ns
    Presença de sequências curtas
    COnteúdo GC
    Presença de sequÇencias repetidas
    
Iremos visualizar os gráficos de qualidade do sequenciamento

In [ ]:
!fastqc data/*.fastq.gz -o 01_quality_control

#Descompactando resultados
!unzip 01_quality_control/*.zip

Vizualizando resultados:

- Resultados R1

![Per base quality][def]

[def]: 01_quality_control/mycotuberculosis_1_fastqc/Images/per_base_quality.png

- Resultados R2

![Per base quality][def]

[def]: 01_quality_control/mycotuberculosis_2_fastqc/Images/per_base_quality.png

- É possível visualizar que poucas leituras apresentam Phred baixo (menor que 20) indicando que o sequenciamento gerou leituras de boa qualidade e de tamanho aceitável (~150 pb)

## Avaliando a qualidade das leituras com multiqc

- O multiqc gera reports em HTML mais robustos que podem ser visualizados no navegador

In [5]:
!multiqc --dirs 01_quality_control -o 01_quality_control


/// ]8;id=174967;https://multiqc.info\MultiQC]8;;\ 🔍 v1.33

     update_config | Prepending directory to sample names
       file_search | Search path: /MP_Data/Dados_David/genomic_pipeline/01_quality_control
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 56/56  lm  
            fastqc | Found 1 reports
     write_results | Data        : 01_quality_control/multiqc_data
     write_results | Report      : 01_quality_control/multiqc_report.html
           multiqc | MultiQC complete


# Remoção de adaptadores

- Nas etapas anteriores apenas se visualizou como estava o estado da qualidade das leituras do sequenciamento
- Nessa etapa é feito o controle de qualidade de fato removendo as leituras de baixa qualidade e os adaptadores

In [6]:
#Descompactando leituras
!gunzip data/mycotuberculosis_1.fastq.gz
!gunzip data/mycotuberculosis_2.fastq.gz

In [ ]:
#Criar diretório para os outputs
!mkdir 01_quality_control/trimmed

#AdapterRemoval
!AdapterRemoval --file1 data/mycotuberculosis_1.fastq \
    --file2 data/mycotuberculosis_2.fastq \
    --threads 10 \
    --basename 01_quality_control/trimmed/mycotuberculosis \
    --trimns \
    --trimqualities \
    --minquality 30 \
    --minlength 50 \
    --collapse


### Explicando os comandos

- --file1 data/mycotuberculosis_1.fastq
        Define o arquivo FASTQ forward (R1), contendo as leituras do primeiro par de sequenciamento.
- --file2 data/mycotuberculosis_2.fastq
        Define o arquivo FASTQ reverse (R2), contendo as leituras do segundo par (paired-end).
- --threads 10
        Define o uso de 10 threads (núcleos de CPU) para acelerar o processamento das leituras.
- --basename 01_quality_control/trimmed/mycotuberculosis
        Define o prefixo e o diretório dos arquivos de saída gerados pelo AdapterRemoval.
- --trimns
        Remove bases desconhecidas (N) das extremidades das leituras, melhorando a qualidade das sequências.
- --trimqualities
        Remove bases de baixa qualidade nas extremidades das reads com base nos scores Phred.
- --minquality 30
        Define qualidade mínima Phred 30 (99,9% de precisão), removendo bases abaixo desse valor.
- --minlength 50
        Remove leituras com menos de 50 bases após o trimming, evitando sequências muito curtas para análises downstream.
- --collapse
        Junta reads paired-end que possuem sobreposição, gerando uma única sequência mais longa e de maior qualidade.

# Avaliando a qualidade novamente

- Aqui avalia-se novamente a qualidade afim de observar como estão os dados após a remoção de leituras de baixa qualidade

In [ ]:
#Criar diretorio para output
!mkdir 01_quality_control/trimmed/fastqc

!fastqc 01_quality_control/trimmed/mycotuberculosis.pair*.truncated -o 01_quality_control/trimmed/fastqc

![Per base quality][def]

[def]: 01_quality_control/trimmed/fastqc/mycotuberculosis.pair1.truncated_fastqc/Images/per_base_quality.png

![Per base quality][def]

[def]: 01_quality_control/trimmed/fastqc/mycotuberculosis.pair2.truncated_fastqc/Images/per_base_quality.png

# Montagem

- Aqui as leituras serão usadas para montar sequencias contíguas maiores chamadas de contigs
- Será usado o software spades

In [ ]:
#!mkdir 02_assembly/spades

#running spades
!spades.py -1 01_quality_control/trimmed/mycotuberculosis.pair1.truncated.fastq \
    -2 01_quality_control/trimmed/mycotuberculosis.pair2.truncated.fastq \
    -o 02_assembly/spades \
    -t 20

## Analisando a montagem

- Com o software quast é possível extrair alguns dados básicos da montagem como:
    - Métricas de Contigs e Scaffolds: Número total de contigs/scaffolds, tamanho total do genoma montado, N50, L50, tamanho mínimo e       máximo, e número de contigs com mais de 500bp ou outro tamanho definido.
    - Comparação com Referência: Se uma referência estiver disponível, o QUAST identifica erros estruturais, tais como:
    - Misassemblies (Montagens incorretas): Número de quebras, inversões, translocações e deslocamentos.
    - Variações estruturais: Diferenças entre a montagem e a referência.
    - Contigs desalinhados: Contigs que não mapeiam para a referência.
    - Métricas de Consenso: Conteúdo GC (GC%) e número de "N"s por 100 kbp.
    - Relatórios e Visualização: Gera relatórios em texto, gráficos estáticos (curvas contig, GC) e gráficos interativos em HTML.

In [ ]:
!quast 02_assembly/spades/contigs.fasta -o 02_assembly/quast

- Os resultados podem ser visualizados completamente no navegador porém aqui vamos ver apenas uma parte do que é gerado;
- Os dados pricnipais da montagem ficam no arquivo: report.tsv

In [3]:
import pandas as pd

quast_results = pd.read_csv('02_assembly/quast/report.tsv',sep='\t')
quast_results 

,Assembly,contigs
0,# contigs (>= 0 bp),511.00
1,# contigs (>= 1000 bp),94.00
2,# contigs (>= 5000 bp),61.00
3,# contigs (>= 10000 bp),56.00
4,# contigs (>= 25000 bp),45.00
5,# contigs (>= 50000 bp),35.00
6,Total length (>= 0 bp),4473276.00
7,Total length (>= 1000 bp),4358381.00
8,Total length (>= 5000 bp),4288383.00
9,Total length (>= 10000 bp),4251216.00


- Dentre as métricas mais importantes de se avaliar estão o tamanho do genoma 
    Caso seja muito diferente do que se espera para a espécie bacteriana podeser indicativo de contaminação
- O mesmo vale para o conteúdo GC
-  N's per 100 kbp indica a presença de Ns (caso se esteja trabalhando com contigs Ns não devem ser esperados)
- N50 e L50 avaliam a fragmentação da montagem.
    O N50 é o comprimento da maior sequência onde a soma dos contigs, ordenados do maior para o menor, atinge 50% do total da montagem. O L50 é o número mínimo de contigs que, somados, cobrem essa mesma metade da montagem.